# Testing the implementation of biLSTM

## Load required libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Bidirectional, Dense, Dropout, Masking, Input
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from scikeras.wrappers import KerasClassifier

In [2]:
import sklearn
import scikeras

print("scikit-learn version:", sklearn.__version__)  # Should be at least 1.0
print("scikeras version:", scikeras.__version__)  # Should be at least 0.11

import tensorflow as tf
import mediapipe as mp

print(tf.__version__)  # Should print TensorFlow version
print(mp.__version__)  # Should print mediapipe version



scikit-learn version: 1.3.0
scikeras version: 0.12.0
2.15.0
0.10.21


# Preprocessing

In [13]:
data = np.load("../squat_sequences.npz")
X_data = data["X"]
y_labels = data["y"]

# Now split and train:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_data, y_labels, test_size=0.2, stratify=y_labels, random_state=42
)

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)


(157, 189, 3) (40, 189, 3) (157,) (40,)


# Defined model Function

In [4]:

def create_bilstm_model(input_shape, lstm_units=64, dropout_rate=0.3, learning_rate=0.001, **kwargs):
    """Creates and compiles a BiLSTM model with given hyperparameters."""
    
    model = Sequential([
        Input(shape=input_shape),
        Masking(mask_value=0.0),
        Bidirectional(LSTM(lstm_units, return_sequences=True)),
        Dropout(dropout_rate),
        Bidirectional(LSTM(lstm_units // 2)),
        Dropout(dropout_rate),
        Dense(16, activation='relu'),
        Dense(1, activation='sigmoid')  # Binary classification
    ])
    
    optimizer = Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    
    return model

# Define Hyperparameter Grid

In [5]:
param_grid = {
    'model__lstm_units': [64, 128],  # Reduce choices
    'model__dropout_rate': [0.2, 0.3],  
    'model__learning_rate': [0.001, 0.0005]  
}

In [6]:
test_model = KerasClassifier(
    model=create_bilstm_model,
    model__input_shape=(X_train.shape[1], X_train.shape[2]),
    model__lstm_units=64,
    model__dropout_rate=0.3,
    model__learning_rate=0.001,
    epochs=1,
    batch_size=16,
    verbose=0
)



# Ensure y_train is a 1D array of integers
y_train = y_train.astype(int).ravel()

# Train the model on a small subset to test
test_model.fit(X_train[:10], y_train[:10])


KerasClassifier(
	model=<function create_bilstm_model at 0x0000016BF67468E0>
	build_fn=None
	warm_start=False
	random_state=None
	optimizer=rmsprop
	loss=None
	metrics=None
	batch_size=16
	validation_batch_size=None
	verbose=0
	callbacks=None
	validation_split=0.0
	shuffle=True
	run_eagerly=False
	epochs=1
	model__input_shape=(189, 3)
	model__lstm_units=64
	model__dropout_rate=0.3
	model__learning_rate=0.001
	class_weight=None
)

# Nested Cross-Validation Implementation

In [ ]:

# Convert dataset to NumPy arrays
X = np.array(X_train)
y = np.array(y_train)

# Define 5-Fold Cross-Validation
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Store results
best_params_per_fold = []
validation_scores = []

for train_idx, val_idx in outer_cv.split(X, y):
    X_train_fold, X_val_fold = X[train_idx], X[val_idx]
    y_train_fold, y_val_fold = y[train_idx], y[val_idx]

    model = KerasClassifier(
        model=create_bilstm_model,
        model__input_shape=(X_train.shape[1], X_train.shape[2]),
        model__lstm_units=64,
        model__dropout_rate=0.3,
        model__learning_rate=0.001,
        epochs=30,
        batch_size=16,
        verbose=0,
        metrics=['accuracy']  # ✅ Explicitly set metrics here
    )


    # Inner Grid Search
    grid_search = GridSearchCV(
        estimator=model, 
        param_grid=param_grid, 
        cv=3, 
        scoring='accuracy', 
        n_jobs=1  # Force single process
    )

    grid_search.fit(X_train_fold, y_train_fold)

    # Get the best parameters and score
    best_params = grid_search.best_params_
    best_params_per_fold.append(best_params)

    best_model = grid_search.best_estimator_
    val_accuracy = best_model.score(X_val_fold, y_val_fold)
    validation_scores.append(val_accuracy)

    print(f"🔍 Fold Validation Accuracy: {val_accuracy:.4f} - Best Params: {best_params}")

# Final evaluation
print(f"📊 Nested CV Accuracy: {np.mean(validation_scores):.4f} ± {np.std(validation_scores):.4f}")
print("📝 Best parameters per fold:", best_params_per_fold)


ValueError: 
All the 24 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
24 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\Marteinn\Desktop\Software-Tech\Semester6\thesis\BScThesis\mediapipe-env\Lib\site-packages\sklearn\model_selection\_validation.py", line 732, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\Marteinn\Desktop\Software-Tech\Semester6\thesis\BScThesis\mediapipe-env\Lib\site-packages\scikeras\wrappers.py", line 1491, in fit
    super().fit(X=X, y=y, sample_weight=sample_weight, **kwargs)
  File "c:\Users\Marteinn\Desktop\Software-Tech\Semester6\thesis\BScThesis\mediapipe-env\Lib\site-packages\scikeras\wrappers.py", line 760, in fit
    self._fit(
  File "c:\Users\Marteinn\Desktop\Software-Tec

# Evaluation

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

# Get predictions and convert probabilities to binary labels
y_pred_probs = test_model.predict(X_test)
y_pred = (y_pred_probs > 0.5).astype("int")

# Print classification report
print(classification_report(y_test, y_pred))

# Compute ROC-AUC score
roc_auc = roc_auc_score(y_test, y_pred_probs)
print(f"🏆 ROC-AUC Score: {roc_auc:.4f}")